# Instance Segmentation

Add a tiny mask branch to a Faster R-CNN detector and you have instance segmentation. The hard part is RoIAlign, and it is harder than it looks.

## Problem Definition

Semantic segmentation gives you one mask per class. Instance segmentation gives you one mask per object, even when two objects share a class.

The hard engineering problem is sampling: how do you crop a fixed-size feature region out of a proposal box whose corners do not align with pixel boundaries? Getting that wrong costs tenths of a mAP point everywhere. RoIAlign is the answer.

## Basic Concept

### The architecture

```mermaid
flowchart TD
    IMG["Input"] --> BB["ResNet<br/>backbone"]
    BB --> FPN["Feature<br/>Pyramid Network"]
    FPN --> RPN["Region<br/>Proposal<br/>Network"]
    FPN --> RA["RoIAlign"]
    RPN -->|"top-K proposals"| RA
    RA --> BH["Box head<br/>(class + refine)"]
    RA --> MH["Mask head<br/>(14x14 conv)"]
    BH --> NMS["NMS"]
    MH --> NMS
    NMS --> OUT["boxes +<br/>classes + masks"]

    style BB fill:#dbeafe,stroke:#2563eb
    style FPN fill:#fef3c7,stroke:#d97706
    style RPN fill:#fecaca,stroke:#dc2626
    style OUT fill:#dcfce7,stroke:#16a34a
```

Five pieces to understand:

1. **Backbone** — ResNet-50 or ResNet-101 trained on ImageNet. Produces a hierarchy of feature maps at strides 4, 8, 16, 32.
2. **FPN (Feature Pyramid Network)** — top-down + lateral connections that give every level C channels of semantic-rich features. Detection queries the FPN level matching the object size.
3. **RPN (Region Proposal Network)** — a small conv head that, at every anchor position, predicts "is there an object here?" and "how do I refine the box?". Produces ~1000 proposals per image.
4. **RoIAlign** — samples a fixed-size (e.g. 7x7) feature patch from any box on any FPN level. Bilinear sampling, no quantisation.
5. **Heads** — two-layer box head that refines the box and picks a class, plus a small conv head that outputs a `28x28` binary mask for each proposal.


# Build your Own

In [1]:
import torch
import torch.nn.functional as F

def roi_align_single(features, box, output_size=7, spatial_scale=1/16.0):
    C, H, W = features.shape
    x1, y1, x2, y2 = [c * spatial_scale - 0.5 for c in box]
    bin_w = (x2 - x1) / output_size
    bin_h = (y2 - y1) / output_size

    grid_y = torch.linspace(y1 + bin_h / 2, y2 - bin_h / 2, output_size)
    grid_x = torch.linspace(x1 + bin_w / 2, x2 - bin_w / 2, output_size)
    yy, xx = torch.meshgrid(grid_y, grid_x, indexing="ij")

    gx = 2 * (xx + 0.5) / W - 1
    gy = 2 * (yy + 0.5) / H - 1
    grid = torch.stack([gx, gy], dim=-1).unsqueeze(0)
    
    sampled = F.grid_sample(features.unsqueeze(0), grid, mode="bilinear", align_corners=False)

    return sampled.squeeze(0)

In [2]:
from torchvision.ops import roi_align

feature = torch.randn(1, 16, 50, 50)
boxes = torch.tensor([[0, 10, 20, 100, 90]], dtype=torch.float32)

ours = roi_align_single(feature[0], boxes[0, 1:].tolist(), output_size=7, spatial_scale=1/16.0)

theirs = roi_align(feature, boxes, output_size=(7, 7), spatial_scale=1/16.0, sampling_ratio=1, aligned=True)[0]

print(f"shape ours:   {tuple(ours.shape)}")
print(f"shape theirs: {tuple(theirs.shape)}")
print(f"max|diff|:    {(ours - theirs).abs().max().item():.3e}")

shape ours:   (16, 7, 7)
shape theirs: (16, 7, 7)
max|diff|:    2.623e-06


## Load a  Pretrained Mask R-CNN

In [3]:
import torch
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights

model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
model.eval()
print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print(f"classes (including background): {len(model.roi_heads.box_predictor.cls_score.out_features * [0])}")

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /Users/keyficller/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth


100%|██████████| 177M/177M [00:07<00:00, 24.2MB/s] 


params: 46,359,409
classes (including background): 91


In [4]:
with torch.no_grad():
    x = torch.randn(3, 400, 600)
    predictions = model([x])
p = predictions[0]
print(f"boxes:  {tuple(p['boxes'].shape)}")
print(f"labels: {tuple(p['labels'].shape)}")
print(f"scores: {tuple(p['scores'].shape)}")
print(f"masks:  {tuple(p['masks'].shape)}")

boxes:  (61, 4)
labels: (61,)
scores: (61,)
masks:  (61, 1, 400, 600)


## Fine-tune heads: shape instances (class = geometry)

Same setup as the semantic-segmentation demo, but each object is a separate instance (own box + mask). Color is random; class follows shape.

| id | class |
|----|--------|
| 0 | background (implicit) |
| 1 | circle |
| 2 | square |
| 3 | triangle |

Pretrained `maskrcnn_resnet50_fpn_v2` → replace box + mask predictors → freeze backbone/RPN → train RoI heads.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

NUM_CLASSES = 4  # bg + circle + square + triangle
CLASS_NAMES = {1: "circle", 2: "square", 3: "triangle"}


class ShapeInstanceDataset(Dataset):
    """Random-colored shapes; label/mask keyed by geometry. Multiple instances OK."""

    def __init__(self, n=48, size=192, seed=0):
        self.n = n
        self.size = size
        self.rng = random.Random(seed)

    def __len__(self):
        return self.n

    def _rand_rgb(self, avoid=None, min_delta=120):
        while True:
            c = (self.rng.randint(0, 255), self.rng.randint(0, 255), self.rng.randint(0, 255))
            if avoid is None or sum(abs(a - b) for a, b in zip(c, avoid)) >= min_delta:
                return c

    def __getitem__(self, idx):
        s = self.size
        bg = self._rand_rgb()
        img = Image.new("RGB", (s, s), bg)
        draw = ImageDraw.Draw(img)

        boxes, labels, masks = [], [], []
        # 1–2 instances per shape class → can share a class (instance seg)
        shape_plan = []
        for class_id in (1, 2, 3):
            for _ in range(self.rng.randint(1, 2)):
                shape_plan.append(class_id)
        self.rng.shuffle(shape_plan)

        for class_id in shape_plan:
            color = self._rand_rgb(avoid=bg)
            side = self.rng.randint(s // 7, s // 3)
            x0 = self.rng.randint(0, s - side)
            y0 = self.rng.randint(0, s - side)
            x1, y1 = x0 + side, y0 + side

            inst = Image.new("L", (s, s), 0)
            d_inst = ImageDraw.Draw(inst)
            if class_id == 1:
                draw.ellipse([x0, y0, x1, y1], fill=color)
                d_inst.ellipse([x0, y0, x1, y1], fill=1)
            elif class_id == 2:
                draw.rectangle([x0, y0, x1, y1], fill=color)
                d_inst.rectangle([x0, y0, x1, y1], fill=1)
            else:
                pts = [(x0, y1), (x1, y1), ((x0 + x1) // 2, y0)]
                draw.polygon(pts, fill=color)
                d_inst.polygon(pts, fill=1)

            m = np.array(inst, dtype=np.uint8)
            ys, xs = np.where(m > 0)
            if len(xs) == 0:
                continue
            boxes.append([float(xs.min()), float(ys.min()), float(xs.max()) + 1, float(ys.max()) + 1])
            labels.append(class_id)
            masks.append(m)

        image = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.tensor(np.stack(masks), dtype=torch.uint8),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def build_shape_maskrcnn(num_classes=NUM_CLASSES):
    model = maskrcnn_resnet50_fpn_v2(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

    for p in model.parameters():
        p.requires_grad = False
    # freeze backbone + RPN; train RoI box/mask heads
    for p in model.roi_heads.parameters():
        p.requires_grad = True
    return model


def train_shape_maskrcnn(epochs=8, steps_per_epoch=12, batch_size=2, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    ds = ShapeInstanceDataset(n=steps_per_epoch * batch_size, size=192, seed=0)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    model = build_shape_maskrcnn().to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for images, targets in loader:
            images = [im.to(device) for im in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
            n += 1
        print(f"epoch {epoch + 1}/{epochs}  loss={total / max(n, 1):.4f}")
    return model, device


def show_instance_pred(model, device, seed=99, score_thresh=0.5):
    ds = ShapeInstanceDataset(n=1, size=192, seed=seed)
    image, target = ds[0]
    model.eval()
    with torch.no_grad():
        pred = model([image.to(device)])[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, title in zip(axes, ["ground truth", "prediction"]):
        ax.imshow(image.permute(1, 2, 0).numpy())
        ax.set_title(title)
        ax.axis("off")

    def _overlay(ax, boxes, labels, masks, scores=None):
        for i in range(len(boxes)):
            if scores is not None and scores[i] < score_thresh:
                continue
            m = masks[i]
            if m.ndim == 3:
                m = m[0]
            color = np.array(plt.cm.tab10(int(labels[i]) % 10)[:3])
            rgba = np.zeros((*m.shape, 4))
            rgba[m.cpu().numpy() > 0.5] = (*color, 0.45)
            ax.imshow(rgba)
            x1, y1, x2, y2 = boxes[i].tolist()
            ax.add_patch(
                mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.5)
            )
            name = CLASS_NAMES.get(int(labels[i]), str(int(labels[i])))
            tag = f"{name}" if scores is None else f"{name} {scores[i]:.2f}"
            ax.text(x1, max(y1 - 3, 0), tag, color="white", fontsize=8, backgroundcolor=(0, 0, 0, 0.5))

    _overlay(axes[0], target["boxes"], target["labels"], target["masks"])
    _overlay(axes[1], pred["boxes"].cpu(), pred["labels"].cpu(), pred["masks"].cpu(), pred["scores"].cpu())
    keep = pred["scores"] >= score_thresh
    print(f"GT instances: {len(target['boxes'])}")
    print(f"pred above {score_thresh}: {int(keep.sum())} / {len(pred['scores'])}")
    plt.tight_layout()
    plt.show()


model, device = train_shape_maskrcnn()
show_instance_pred(model, device)